# ArgLLM baseline (DF-QuAD) — DeepSeek-V3 on StrategyClaim (500 claims, with the annotators' facts)

Authors' prompts from ArgLLM `prompt.py` (`ArgumentMiningPrompts.opro` for arguments, `UncertaintyEvaluatorPrompts.analyst` for strengths), with the dataset facts added to every prompt, temperature 0.7, top-p 0.95; DF-QuAD from the authors' Uncertainpy code.

The second code cell runs all claims; the third asks again (longer answer, up to three times) for every strength that could not be read, and gives the final numbers reported in the paper (0.5 Base 72.6, Est. Base 87.0).

In [ ]:
!pip install -q openai datasets scikit-learn

import os, re, sys, time, pickle
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

key = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
client = OpenAI(api_key=key, base_url="https://api.deepseek.com",
                timeout=60, max_retries=2)
MODEL_NAME = "deepseek-chat"

WORK_DIR = Path("/kaggle/working/baselines_deepseek")
WORK_DIR.mkdir(parents=True, exist_ok=True)

repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q "
              "https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk(repo + "/Datasets/StrategyQA/Experiment")
df = df.to_pandas().reset_index(drop=True)


def format_facts(f):
    if isinstance(f, (list, tuple, np.ndarray)):
        return "\n".join("- " + str(x) for x in f)
    return "" if f is None else str(f)


claims = df["claim"].astype(str).tolist()
facts = df["facts"].apply(format_facts).tolist()
gold = df["valid"].apply(lambda v: int(bool(v))).tolist()

sys.path.insert(0, repo)
import Uncertainpy.src.uncertainpy.gradual as grad
agg_f = grad.semantics.modular.ProductAggregation()
inf_f = grad.semantics.modular.LinearInfluence(conservativeness=1)

print(MODEL_NAME, "| claims:", len(claims), "| true:", sum(gold))
print("first facts:\n", facts[0])

deepseek-chat | claims: 500 | true: 250
first facts:
 ['Reza Shah, the founder of the Imperial state of Iran, was a Twelver Shia Muslim.'
 'Twelver Shia Muslims believe that there are 12 Imams.'
 'Jesus is typically represented as having 12 disciples.']


In [ ]:
def call(prompt, temperature, max_tokens, top_p=None):
    kw = dict(model=MODEL_NAME, temperature=temperature,
              max_tokens=max_tokens,
              messages=[{"role": "user", "content": prompt}])
    if top_p is not None:
        kw["top_p"] = top_p
    last = None
    for a in range(1, 9):
        try:
            r = client.chat.completions.create(**kw)
            t = r.choices[0].message.content
            if t:
                return t.strip()
            raise ValueError("empty")
        except Exception as e:
            last = e
            busy = any(c in str(e) for c in ["429", "502", "503", "504"])
            time.sleep(20 * a if busy else 3 * a)
    raise RuntimeError(f"failed: {last}")


def am_prompt(claim, fact, support):
    s = "supporting" if support else "attacking"
    v = "support" if support else "attack"
    return (f"Facts:\n{fact}\n\n"
            f"Please provide a single short argument {s} the following "
            "claim. Construct the argument so it refers to the truthfulness "
            "of the claim. Only provide an argument if you think there is a "
            f"valid and convincing {v} for this claim (there is a non-zero "
            "probability that the claim is true), otherwise return: N/A.\n"
            f"Claim: {claim}\n"
            "Now take a deep breath and come up with an argument.\n"
            "Argument:")


CALIB = ("definitely valid and values in between indicating various levels "
         "of uncertainty. Your estimates should be well-calibrated, so feel "
         "free to err on the side of caution and output moderate "
         "probabilities if you are not completely sure in your assessment. "
         "Please respond in the following form:\n\nLikelihood: The "
         "predicted likelihood that the considered ")


def ue_arg_prompt(claim, fact, arg, support):
    return (f"Facts:\n{fact}\n\n"
            "You are an analyst evaluating the validity and relevance of "
            f'arguments. For the argument:\n\nArgument: "{arg}"\n\nplease '
            "give your confidence that the argument presents a compelling "
            f"case {'in favour of' if support else 'against'} the statement:"
            f'\n\nStatement: "{claim}"\n\nYour assessment should be based on '
            f"how well the argument {'supports' if support else 'refutes'} "
            "the considered statement as well as the correctness, accuracy "
            "and truthfulness of the given argument. Your response should be "
            "between 0% and 100% with 0% indicating that the considered "
            "argument is definitely invalid, 100% indicating that the "
            "considered argument is " + CALIB + "argument is valid")


def ue_topic_prompt(claim, fact):
    return (f"Facts:\n{fact}\n\n"
            "You are an analyst evaluating the validity of statements. For "
            f'the statement:\n\nStatement: "{claim}"\n\nplease give your '
            "confidence that the statement is correct, accurate and "
            "truthful. Your response should be between 0% and 100% with 0% "
            "indicating that the considered statement is definitely invalid, "
            "100% indicating that the considered statement is "
            + CALIB + "statement is valid")


def gen_argument(claim, fact, support):
    t = call(am_prompt(claim, fact, support), 0.7, 150, top_p=0.95)
    if "N/A" in t or "n/a" in t:
        return "N/A"
    return t.replace("Argument:", "").strip().strip('"')


PARSE_FAILS = 0
def get_lik(prompt):
    global PARSE_FAILS
    t = call(prompt, 0.7, 60, top_p=0.95)
    m = (re.search(r"Likelihood:\s*(\d{1,3})\s*%?", t)
         or re.search(r"(\d{1,3})\s*%", t))
    if m:
        return min(int(m.group(1)), 100) / 100
    PARSE_FAILS += 1
    return 0.5


def strength(v, est):
    tau = v["topic_base"] if est else 0.5
    bag = grad.BAG()
    c = grad.Argument("db0", v["claim"], float(tau))
    bag.arguments["db0"] = c
    if v["sup"] != "N/A":
        bag.add_support(grad.Argument("S1", "s", float(v["sup_base"])), c)
    if v["att"] != "N/A":
        bag.add_attack(grad.Argument("A1", "a", float(v["att_base"])), c)
    grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)


QCK = WORK_DIR / "dfquad_authors_facts_ckpt.pkl"
qres = pickle.load(open(QCK, "rb")) if QCK.exists() else {}
print("already done:", len(qres))

for i in range(len(claims)):
    if i in qres:
        continue
    c, f = claims[i], facts[i]
    try:
        v = {"claim": c, "gold": gold[i]}
        v["sup"] = gen_argument(c, f, True)
        v["att"] = gen_argument(c, f, False)
        v["sup_base"] = (get_lik(ue_arg_prompt(c, f, v["sup"], True))
                         if v["sup"] != "N/A" else None)
        v["att_base"] = (get_lik(ue_arg_prompt(c, f, v["att"], False))
                         if v["att"] != "N/A" else None)
        v["topic_base"] = get_lik(ue_topic_prompt(c, f))
        v["s_05"] = strength(v, False)
        v["s_est"] = strength(v, True)
        qres[i] = v
    except Exception as e:
        print("error", i, str(e)[:60])
    pickle.dump(qres, open(QCK, "wb"))
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(claims)}")


def report(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
    return {"method": name, "n": len(pred),
            "accuracy": round(accuracy_score(gold, pred) * 100, 1),
            "precision": round(precision_score(gold, pred,
                                               zero_division=0) * 100, 1),
            "recall": round(recall_score(gold, pred,
                                         zero_division=0) * 100, 1),
            "f1": round(f1_score(gold, pred, zero_division=0) * 100, 1),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}


res = pd.DataFrame.from_dict(qres, orient="index").sort_index()
if len(res) < len(claims):
    print(f"only {len(res)}/{len(claims)} done - run this cell again")
else:
    p05 = (res["s_05"] > 0.5).astype(int).tolist()
    pest = (res["s_est"] > 0.5).astype(int).tolist()
    rows = [report("DF-QuAD 0.5 Base", p05),
            report("DF-QuAD Est. Base", pest)]
    print("\n=== DeepSeek - StrategyClaim - DF-QuAD with facts ===")
    print(pd.DataFrame(rows).to_string())
    print("N/A supporters:", (res.sup == "N/A").sum(),
          "| N/A attackers:", (res.att == "N/A").sum(),
          "| parse failures (this session):", PARSE_FAILS)

already done: 0
  10/500
  20/500
  30/500
  40/500
  50/500
  60/500
  70/500
  80/500
  90/500
  100/500
  110/500
  120/500
  130/500
  140/500
  150/500
  160/500
  170/500
  180/500
  190/500
  200/500
  210/500
  220/500
  230/500
  240/500
  250/500
  260/500
  270/500
  280/500
  290/500
  300/500
  310/500
  320/500
  330/500
  340/500
  350/500
  360/500
  370/500
  380/500
  390/500
  400/500
  410/500
  420/500
  430/500
  440/500
  450/500
  460/500
  470/500
  480/500
  490/500
  500/500

=== DeepSeek - StrategyClaim - DF-QuAD with facts ===
              method    n  accuracy  precision  recall    f1   TN  FP   FN   TP
0   DF-QuAD 0.5 Base  500      73.0       93.9    49.2  64.6  242   8  127  123
1  DF-QuAD Est. Base  500      81.6       95.9    66.0  78.2  243   7   85  165
N/A supporters: 235 | N/A attackers: 52 | parse failures (this session): 187


In [ ]:
def read_lik(t):
    t = t or ""
    m = re.findall(r"Likelihood\W*(\d{1,3}(?:\.\d+)?)\s*%", t, re.I)
    if not m:
        m = re.findall(r"(\d{1,3}(?:\.\d+)?)\s*%", t)
    if m:
        return min(float(m[-1]), 100) / 100
    m = re.findall(r"Likelihood\W*(0?\.\d+|1\.0+|0|1)\b", t, re.I)
    if m:
        return float(m[-1])
    return None


def get_lik2(prompt):
    for _ in range(3):
        v = read_lik(call(prompt, 0.7, 400, top_p=0.95))
        if v is not None:
            return v
    return None


fixed, still_bad = 0, 0
for k, (i, v) in enumerate(sorted(qres.items()), 1):
    c, f = claims[i], facts[i]
    jobs = []
    if v["sup"] != "N/A" and v["sup_base"] == 0.5:
        jobs.append(("sup_base", ue_arg_prompt(c, f, v["sup"], True)))
    if v["att"] != "N/A" and v["att_base"] == 0.5:
        jobs.append(("att_base", ue_arg_prompt(c, f, v["att"], False)))
    if v["topic_base"] == 0.5:
        jobs.append(("topic_base", ue_topic_prompt(c, f)))
    for key_, p in jobs:
        new = get_lik2(p)
        if new is None:
            still_bad += 1
        else:
            v[key_] = new
            fixed += 1
    if jobs:
        v["s_05"] = strength(v, False)
        v["s_est"] = strength(v, True)
        pickle.dump(qres, open(QCK, "wb"))
    if k % 50 == 0:
        print(f"  {k}/500 | fixed {fixed} | still unreadable {still_bad}")

print(f"\nfixed: {fixed} | still unreadable: {still_bad}")

res = pd.DataFrame.from_dict(qres, orient="index").sort_index()
p05 = (res["s_05"] > 0.5).astype(int).tolist()
pest = (res["s_est"] > 0.5).astype(int).tolist()
rows = [report("DF-QuAD 0.5 Base", p05),
        report("DF-QuAD Est. Base", pest)]
print("\n=== DeepSeek - StrategyClaim - DF-QuAD with facts (repaired) ===")
print(pd.DataFrame(rows).to_string())

  50/500 | fixed 21 | still unreadable 0
  100/500 | fixed 36 | still unreadable 0
  150/500 | fixed 55 | still unreadable 0
  200/500 | fixed 79 | still unreadable 0
  250/500 | fixed 96 | still unreadable 0
  300/500 | fixed 120 | still unreadable 0
  350/500 | fixed 133 | still unreadable 0
  400/500 | fixed 156 | still unreadable 0
  450/500 | fixed 173 | still unreadable 0
  500/500 | fixed 195 | still unreadable 0

fixed: 195 | still unreadable: 0

=== DeepSeek - StrategyClaim - DF-QuAD with facts (repaired) ===
              method    n  accuracy  precision  recall    f1   TN  FP   FN   TP
0   DF-QuAD 0.5 Base  500      72.6       93.8    48.4  63.9  242   8  129  121
1  DF-QuAD Est. Base  500      87.0       96.0    77.2  85.6  242   8   57  193


In [ ]:
import os, shutil, pickle
import pandas as pd
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

# 1) final results (all 500 claims, repaired)
final_pkl = WORK_DIR / "dfquad_deepseek_strategyclaim_final.pkl"
pickle.dump(qres, open(final_pkl, "wb"))

# 2) readable table: one row per claim
res = pd.DataFrame.from_dict(qres, orient="index").sort_index()
res["pred_05"] = (res["s_05"] > 0.5).astype(int)
res["pred_est"] = (res["s_est"] > 0.5).astype(int)
res.to_csv(WORK_DIR / "dfquad_deepseek_strategyclaim_final.csv",
           index_label="index")

# 3) one zip with everything in the folder
shutil.make_archive("dfquad_deepseek_final", "zip", WORK_DIR)

print("claims saved:", len(res))
print("zip size:", os.path.getsize("dfquad_deepseek_final.zip") // 1024, "KB")
display(FileLink("dfquad_deepseek_final.zip"))

claims saved: 500
zip size: 231 KB


/kaggle/working/dfquad_deepseek_final.zip